# 基于 MindSpore 的伪造人脸图像检测

**任务：** Real / Fake 二分类  
**基础模型：** SimpleCNN  
**主模型：** ResNet18（随机初始化，从头训练，不使用预训练权重）  
**框架：** MindSpore  
**推荐环境：** 华为云 ModelArts + Ascend NPU

Notebook 流程：环境检查 → 数据处理 → 模型定义 → 训练与验证 → checkpoint 测试 → 实验结果可视化。

> 正式训练默认关闭。需要复现实验时，将训练开关设置为 `True`。

> **运行说明**
>
> - 建议第一次使用时执行 `Kernel → Restart Kernel and Run All Cells`。
> - 提交版训练开关默认均为 `False`，因此 Run All 不会重新训练。
> - 数据、模型和可视化单元已补充必要的本地 import / 默认配置，避免截图中出现的 `BATCH_SIZE`、`nn`、`np`、`plt` 未定义问题。
> - 如需重新训练，再手动将训练开关改为 `True`。
> - 本版已接入 AtomGit 仓库 `qq_627548571/face` 的 `experiment9` 目录。首次运行会按需下载 `archive.zip`、`resnet18_best.ckpt`、`simple_cnn_best.ckpt`；后续检测到本地有效文件后不会重复下载。
> - 仓库中的 3 个文件均为 Git LFS 文件，因此运行环境需要可用的 `git` 与 `git lfs`。



## 1. 环境与参数配置

本实验使用 MindSpore 2.7.2。Notebook 不强制要求 Ascend，因此在 CPU 环境中也可阅读和执行轻量单元；完整训练建议使用 Ascend NPU。

In [ ]:
from pathlib import Path
import json
import time
import random
import warnings

import numpy as np
import matplotlib.pyplot as plt

import mindspore as ms
from mindspore import nn, ops
import mindspore.dataset as ds
import mindspore.dataset.vision as vision

warnings.filterwarnings("ignore", message="The value of the smallest subnormal*")

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 8

CLASS_INDEXING = {"real": 0, "fake": 1}
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

ms.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
ds.config.set_seed(SEED)

print("MindSpore版本：", ms.__version__)
print("当前设备：", ms.get_context("device_target"))
print("设备编号：", ms.get_context("device_id"))

### 1.1 AtomGit 仓库资源准备

本实验所需的大文件统一从下面的 AtomGit 仓库获取：

`https://ai.gitcode.com/qq_627548571/face/tree/main/experiment9`

使用的仓库文件：

- `experiment9/archive.zip`：数据集压缩包；
- `experiment9/resnet18_best.ckpt`：ResNet18 最佳权重；
- `experiment9/simple_cnn_best.ckpt`：SimpleCNN 最佳权重。

代码采用 **Git + Git LFS 按需拉取**。首次运行会下载缺失文件；再次运行时若本地文件已存在且不是 LFS pointer，则直接复用，不重复下载。为避免把仓库其他实验中的大文件一起拉下来，clone 时关闭 LFS 自动 smudge，只对 `experiment9` 的 3 个目标文件执行 `git lfs pull`。


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import zipfile

# ===== AtomGit experiment9 配置 =====
ATOMGIT_REPO_PAGE = "https://ai.gitcode.com/qq_627548571/face/tree/main/experiment9"
ATOMGIT_CLONE_URLS = [
    "https://ai.gitcode.com/qq_627548571/face.git",
    "https://gitcode.com/qq_627548571/face.git",  # 兼容备用地址
]
ATOMGIT_BRANCH = "main"
ATOMGIT_SUBDIR = "experiment9"

# ModelArts 优先使用 /home/ma-user/work；其他环境回退到当前目录
MODELARTS_WORK = Path("/home/ma-user/work")
WORK_ROOT = MODELARTS_WORK if MODELARTS_WORK.exists() else Path.cwd().resolve()

# 仓库缓存目录。只用于下载资源，不影响实验代码目录。
ATOMGIT_REPO_DIR = WORK_ROOT / "atomgit_face_repo"
ATOMGIT_EXPERIMENT_DIR = ATOMGIT_REPO_DIR / ATOMGIT_SUBDIR

ATOMGIT_ARCHIVE = ATOMGIT_EXPERIMENT_DIR / "archive.zip"
ATOMGIT_RESNET_CKPT = ATOMGIT_EXPERIMENT_DIR / "resnet18_best.ckpt"
ATOMGIT_SIMPLECNN_CKPT = ATOMGIT_EXPERIMENT_DIR / "simple_cnn_best.ckpt"

ATOMGIT_REQUIRED_FILES = {
    "archive.zip": ATOMGIT_ARCHIVE,
    "resnet18_best.ckpt": ATOMGIT_RESNET_CKPT,
    "simple_cnn_best.ckpt": ATOMGIT_SIMPLECNN_CKPT,
}

# 数据集解压到项目缓存目录；后续单元直接读取解压后的 train/valid/test。
ATOM_DATA_EXTRACT_DIR = WORK_ROOT / "fake_face_project" / "dataset_raw"


def _run(cmd, cwd=None, env=None, check=True):
    """运行命令；失败时给出完整错误信息。"""
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if check and result.returncode != 0:
        raise RuntimeError(
            "命令执行失败：\n"
            + " ".join(map(str, cmd))
            + "\n\n"
            + result.stdout[-4000:]
        )
    return result


def _git_lfs_available():
    if shutil.which("git") is None:
        return False
    result = _run(["git", "lfs", "version"], check=False)
    return result.returncode == 0


def _is_lfs_pointer(path):
    """判断文件是否仍只是 Git LFS pointer，而不是实际大文件。"""
    path = Path(path)
    if not path.exists() or not path.is_file():
        return False
    try:
        with path.open("rb") as f:
            head = f.read(256)
        return b"version https://git-lfs.github.com/spec/v1" in head
    except OSError:
        return False


def _file_ready(path):
    path = Path(path)
    return path.exists() and path.is_file() and path.stat().st_size > 0 and not _is_lfs_pointer(path)


def ensure_atomgit_experiment9():
    """Clone 仓库，并且只下载 experiment9 需要的 3 个 Git LFS 文件。"""
    if shutil.which("git") is None:
        raise RuntimeError(
            "当前环境未检测到 git。请先在 ModelArts Terminal 安装 git，"
            "然后重新运行本单元。"
        )

    # 1) 准备仓库。clone 时跳过所有 LFS 大文件，避免下载其他 experiment 的资源。
    if not (ATOMGIT_REPO_DIR / ".git").exists():
        if ATOMGIT_REPO_DIR.exists() and any(ATOMGIT_REPO_DIR.iterdir()):
            raise RuntimeError(
                f"{ATOMGIT_REPO_DIR} 已存在但不是 Git 仓库。"
                "请改名/移走该目录，或修改 ATOMGIT_REPO_DIR 后重试。"
            )
        ATOMGIT_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"

        clone_errors = []
        for clone_url in ATOMGIT_CLONE_URLS:
            if ATOMGIT_REPO_DIR.exists():
                shutil.rmtree(ATOMGIT_REPO_DIR)
            print("正在连接 AtomGit：", clone_url)
            result = _run(
                [
                    "git", "clone",
                    "--depth", "1",
                    "--branch", ATOMGIT_BRANCH,
                    clone_url,
                    str(ATOMGIT_REPO_DIR),
                ],
                env=clone_env,
                check=False,
            )
            if result.returncode == 0:
                print("仓库 clone 完成。")
                break
            clone_errors.append((clone_url, result.stdout[-1500:]))
        else:
            details = "\n\n".join(
                f"[{url}]\n{msg}" for url, msg in clone_errors
            )
            raise RuntimeError(
                "AtomGit 仓库 clone 失败。请确认 ModelArts 可以访问仓库，"
                "以及仓库 Clone HTTPS 地址是否与 ATOMGIT_CLONE_URLS 一致。\n\n"
                + details
            )
    else:
        # 已有缓存仓库时只做轻量更新；更新失败仍允许继续使用已有文件。
        print("检测到本地 AtomGit 仓库缓存：", ATOMGIT_REPO_DIR)
        pull_env = os.environ.copy()
        pull_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        result = _run(
            ["git", "pull", "--ff-only", "origin", ATOMGIT_BRANCH],
            cwd=ATOMGIT_REPO_DIR,
            env=pull_env,
            check=False,
        )
        if result.returncode != 0:
            print("仓库更新失败，将继续检查已有缓存文件。")
            print(result.stdout[-1200:])

    # 2) 检查目标文件；缺失或仍为 LFS pointer 时才执行 git lfs pull。
    need_lfs = [
        path for path in ATOMGIT_REQUIRED_FILES.values()
        if not _file_ready(path)
    ]

    if need_lfs:
        if not _git_lfs_available():
            raise RuntimeError(
                "experiment9 的 archive.zip 和 ckpt 文件使用 Git LFS 存储，"
                "但当前环境未检测到 git-lfs。请先在 ModelArts Terminal 安装 git-lfs，"
                "确认 `git lfs version` 可以运行后，再执行本单元。"
            )

        _run(["git", "lfs", "install", "--local"], cwd=ATOMGIT_REPO_DIR)

        include_paths = ",".join(
            f"{ATOMGIT_SUBDIR}/{name}"
            for name in ATOMGIT_REQUIRED_FILES
        )

        print("正在从 experiment9 下载所需 Git LFS 文件：")
        for name in ATOMGIT_REQUIRED_FILES:
            print("  -", name)

        _run(
            ["git", "lfs", "pull", f"--include={include_paths}"],
            cwd=ATOMGIT_REPO_DIR,
        )
    else:
        print("experiment9 所需文件已存在，跳过重复下载。")

    # 3) 最终完整性检查，防止拿到的仍然只是几百字节的 LFS pointer。
    bad = []
    for name, path in ATOMGIT_REQUIRED_FILES.items():
        if not _file_ready(path):
            bad.append(f"{name}: {path}")

    if bad:
        raise RuntimeError(
            "以下仓库文件没有成功下载为实际内容：\n" + "\n".join(bad)
        )

    print("\nAtomGit experiment9 资源准备完成：")
    for name, path in ATOMGIT_REQUIRED_FILES.items():
        print(f"  {name}: {path} ({path.stat().st_size / 1024 / 1024:.2f} MB)")


def _looks_like_dataset_root(path):
    path = Path(path)
    return all((path / split).is_dir() for split in ("train", "valid", "test"))


def find_extracted_dataset_root(base_dir):
    """在常见目录结构中定位包含 train/valid/test 的数据集根目录。"""
    base_dir = Path(base_dir)
    candidates = [
        base_dir / "real_vs_fake" / "real-vs-fake",
        base_dir / "real-vs-fake",
        base_dir / "archive" / "real_vs_fake" / "real-vs-fake",
        base_dir / "dataset" / "real_vs_fake" / "real-vs-fake",
        base_dir,
    ]
    for p in candidates:
        if _looks_like_dataset_root(p):
            return p

    # 只搜索有限目录深度，避免遍历 14 万张图片。
    if base_dir.exists():
        base_depth = len(base_dir.resolve().parts)
        for root, dirs, _files in os.walk(base_dir):
            root_path = Path(root)
            depth = len(root_path.resolve().parts) - base_depth
            if depth > 4:
                dirs[:] = []
                continue
            if _looks_like_dataset_root(root_path):
                return root_path
    return None


def ensure_dataset_extracted():
    """若本地没有可用数据集，则从 experiment9/archive.zip 解压。"""
    existing = find_extracted_dataset_root(ATOM_DATA_EXTRACT_DIR)
    if existing is not None:
        print("检测到已解压数据集：", existing)
        return existing

    if not _file_ready(ATOMGIT_ARCHIVE):
        raise FileNotFoundError(f"数据集压缩包不可用：{ATOMGIT_ARCHIVE}")

    ATOM_DATA_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print("开始解压 archive.zip（文件较大，首次运行需要一些时间）……")

    # 优先使用系统 unzip；没有时回退到 Python zipfile。
    if shutil.which("unzip"):
        _run([
            "unzip", "-q", "-o",
            str(ATOMGIT_ARCHIVE),
            "-d", str(ATOM_DATA_EXTRACT_DIR),
        ])
    else:
        with zipfile.ZipFile(ATOMGIT_ARCHIVE, "r") as zf:
            zf.extractall(ATOM_DATA_EXTRACT_DIR)

    dataset_root = find_extracted_dataset_root(ATOM_DATA_EXTRACT_DIR)
    if dataset_root is None:
        raise RuntimeError(
            "archive.zip 已解压，但没有找到同时包含 train/valid/test 的数据集目录。"
            f"请检查：{ATOM_DATA_EXTRACT_DIR}"
        )

    print("数据集解压完成：", dataset_root)
    return dataset_root


# 执行资源准备。后续路径单元会直接复用这些变量。
ensure_atomgit_experiment9()
ATOMGIT_DATASET_ROOT = ensure_dataset_extracted()

print("\n仓库页面：", ATOMGIT_REPO_PAGE)
print("数据集根目录：", ATOMGIT_DATASET_ROOT)
print("ResNet18 权重：", ATOMGIT_RESNET_CKPT)
print("SimpleCNN 权重：", ATOMGIT_SIMPLECNN_CKPT)


In [ ]:
# 自动定位项目目录。优先使用上一单元从 AtomGit experiment9 准备的数据集；
# 如果没有运行上一单元，则兼容原提交目录结构。

cwd = Path.cwd().resolve()


def locate_dataset(project_dir):
    candidates = [
        project_dir / "dataset" / "real_vs_fake" / "real-vs-fake",
        project_dir / "dataset_raw" / "real_vs_fake" / "real-vs-fake",
        project_dir / "real_vs_fake" / "real-vs-fake",
        project_dir / "real-vs-fake",
    ]
    for p in candidates:
        if (p / "train").exists() and (p / "valid").exists() and (p / "test").exists():
            return p
    return None


PROJECT_DIR = None
SOURCE_ROOT = None

# 1) 优先使用 AtomGit archive.zip 解压后的数据集
atomgit_dataset = globals().get("ATOMGIT_DATASET_ROOT")
if atomgit_dataset is not None:
    atomgit_dataset = Path(atomgit_dataset)
    if (atomgit_dataset / "train").exists() and (atomgit_dataset / "valid").exists() and (atomgit_dataset / "test").exists():
        SOURCE_ROOT = atomgit_dataset
        PROJECT_DIR = Path(globals().get("WORK_ROOT", cwd)) / "fake_face_project"

# 2) 兼容已有本地项目结构
if SOURCE_ROOT is None:
    for candidate in [cwd, cwd.parent, Path("/home/ma-user/work/fake_face_project")]:
        root = locate_dataset(candidate)
        if root is not None:
            PROJECT_DIR = candidate
            SOURCE_ROOT = root
            break

# 3) 仍未找到时给出默认路径；数据处理单元会明确提示不存在
if PROJECT_DIR is None:
    PROJECT_DIR = cwd.parent if cwd.name == "code" else cwd
    SOURCE_ROOT = PROJECT_DIR / "dataset" / "real_vs_fake" / "real-vs-fake"

TRAIN_DIR = SOURCE_ROOT / "train"
VAL_DIR = SOURCE_ROOT / "valid"
TEST_DIR = SOURCE_ROOT / "test"

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RESULT_DIR = PROJECT_DIR / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR：", PROJECT_DIR)
print("SOURCE_ROOT：", SOURCE_ROOT)
print("训练集存在：", TRAIN_DIR.exists())
print("验证集存在：", VAL_DIR.exists())
print("测试集存在：", TEST_DIR.exists())

if globals().get("ATOMGIT_RESNET_CKPT") is not None:
    print("AtomGit ResNet18 checkpoint：", ATOMGIT_RESNET_CKPT)
if globals().get("ATOMGIT_SIMPLECNN_CKPT") is not None:
    print("AtomGit SimpleCNN checkpoint：", ATOMGIT_SIMPLECNN_CKPT)


## 2. 数据读取与预处理

数据集共约 14 万张图像：

- 训练集：100,000（50,000 Real + 50,000 Fake）
- 验证集：20,000（10,000 Real + 10,000 Fake）
- 测试集：20,000（10,000 Real + 10,000 Fake）

标签：`real=0`，`fake=1`。

训练阶段使用随机裁剪、随机水平翻转和归一化；验证/测试阶段使用固定缩放、中心裁剪和归一化。

In [ ]:
from pathlib import Path
import shutil
import mindspore.dataset as ds
import mindspore.dataset.vision as vision

# 本单元可单独运行所需的默认配置
IMAGE_SIZE = globals().get("IMAGE_SIZE", 224)
BATCH_SIZE = globals().get("BATCH_SIZE", 32)
NUM_WORKERS = globals().get("NUM_WORKERS", 8)
CLASS_INDEXING = globals().get("CLASS_INDEXING", {"real": 0, "fake": 1})
MEAN = globals().get("MEAN", [0.485, 0.456, 0.406])
STD = globals().get("STD", [0.229, 0.224, 0.225])

# 若前面的路径单元没有运行，优先复用 AtomGit 解压后的数据集，再使用原项目路径兜底
if "TRAIN_DIR" not in globals():
    PROJECT_DIR = Path("/home/ma-user/work/fake_face_project")
    atomgit_root = globals().get("ATOMGIT_DATASET_ROOT")
    if atomgit_root is not None and Path(atomgit_root).exists():
        SOURCE_ROOT = Path(atomgit_root)
    else:
        SOURCE_ROOT = PROJECT_DIR / "dataset_raw" / "real_vs_fake" / "real-vs-fake"
    TRAIN_DIR = SOURCE_ROOT / "train"
    VAL_DIR = SOURCE_ROOT / "valid"
    TEST_DIR = SOURCE_ROOT / "test"

def clean_jupyter_artifacts(dataset_dir):
    """
    删除 Jupyter 自动产生的 .ipynb_checkpoints，
    防止 MindSpore ImageFolderDataset 将其误认为图像文件。
    """

    dataset_dir = Path(dataset_dir)

    for path in dataset_dir.rglob(
        ".ipynb_checkpoints"
    ):
        if path.is_dir():
            shutil.rmtree(path)

def create_dataset(dataset_dir, training, batch_size=None, num_workers=None):
    """创建独立的 MindSpore Dataset。"""
    if batch_size is None:
        batch_size = BATCH_SIZE
    if num_workers is None:
        num_workers = NUM_WORKERS

    dataset_dir = Path(dataset_dir)

    clean_jupyter_artifacts(
        dataset_dir
    )

    if not dataset_dir.exists():
        raise FileNotFoundError(f"数据目录不存在：{dataset_dir}")

    dataset = ds.ImageFolderDataset(
        dataset_dir=str(dataset_dir),
        class_indexing=CLASS_INDEXING,
        shuffle=training,
        num_parallel_workers=num_workers
    )

    sample_count = int(dataset.get_dataset_size())

    if training:
        image_ops = [
            vision.Decode(),
            vision.RandomResizedCrop(
                size=(IMAGE_SIZE, IMAGE_SIZE),
                scale=(0.8, 1.0),
                ratio=(0.9, 1.1)
            ),
            vision.RandomHorizontalFlip(prob=0.5),
            vision.Rescale(1.0 / 255.0, 0.0),
            vision.Normalize(mean=MEAN, std=STD),
            vision.HWC2CHW()
        ]
    else:
        image_ops = [
            vision.Decode(),
            vision.Resize(256),
            vision.CenterCrop((IMAGE_SIZE, IMAGE_SIZE)),
            vision.Rescale(1.0 / 255.0, 0.0),
            vision.Normalize(mean=MEAN, std=STD),
            vision.HWC2CHW()
        ]

    dataset = dataset.map(
        operations=image_ops,
        input_columns="image",
        num_parallel_workers=num_workers
    )

    dataset = dataset.batch(
        batch_size=batch_size,
        drop_remainder=False
    )

    return dataset, sample_count


if TRAIN_DIR.exists():
    train_ds, train_count = create_dataset(TRAIN_DIR, training=True)
    val_ds, val_count = create_dataset(VAL_DIR, training=False)
    test_ds, test_count = create_dataset(TEST_DIR, training=False)

    print("训练样本：", train_count, "批次数：", train_ds.get_dataset_size())
    print("验证样本：", val_count, "批次数：", val_ds.get_dataset_size())
    print("测试样本：", test_count, "批次数：", test_ds.get_dataset_size())
else:
    print("未检测到数据集：", TRAIN_DIR)

## 3. 基础模型：SimpleCNN

SimpleCNN 作为基线模型，通道数依次为 `3 → 32 → 64 → 128 → 256`，最后通过全局平均池化和全连接层进行二分类。

In [ ]:
import numpy as np
from mindspore import nn

class SimpleCNN(nn.Cell):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.SequentialCell([
            nn.Conv2d(3, 32, 3, stride=1, pad_mode="pad", padding=1, has_bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, 3, stride=1, pad_mode="pad", padding=1, has_bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, 3, stride=1, pad_mode="pad", padding=1, has_bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, 3, stride=1, pad_mode="pad", padding=1, has_bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU()
        ])

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.classifier = nn.Dense(256, num_classes)

    def construct(self, x):
        x = self.features(x)
        x = self.avg_pool(x)
        x = self.flatten(x)
        return self.classifier(x)


simple_cnn = SimpleCNN(num_classes=2)
simple_params = sum(int(np.prod(p.shape)) for p in simple_cnn.trainable_params())

print("SimpleCNN创建完成")
print("可训练参数量：", simple_params)

## 4. 主模型：ResNet18

ResNet18 使用残差连接 `y = F(x) + x`，通过快捷路径保留原始信息并改善深层网络的梯度传播。

本项目使用 4 个残差阶段，每个阶段包含 2 个 BasicBlock，通道数依次为 64、128、256、512。

In [ ]:
import numpy as np
from mindspore import nn

class BasicBlock(nn.Cell):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3,
            stride=stride,
            pad_mode="pad",
            padding=1,
            has_bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3,
            stride=1,
            pad_mode="pad",
            padding=1,
            has_bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.use_downsample = stride != 1 or in_channels != out_channels

        if self.use_downsample:
            self.downsample = nn.SequentialCell([
                nn.Conv2d(
                    in_channels, out_channels, 1,
                    stride=stride,
                    pad_mode="valid",
                    has_bias=False
                ),
                nn.BatchNorm2d(out_channels)
            ])

    def construct(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.use_downsample:
            identity = self.downsample(x)

        out = out + identity
        return self.relu(out)


class ResNet18(nn.Cell):
    def __init__(self, num_classes=2):
        super().__init__()

        self.current_channels = 64

        self.stem = nn.SequentialCell([
            nn.Conv2d(
                3, 64, 7,
                stride=2,
                pad_mode="pad",
                padding=3,
                has_bias=False
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, pad_mode="same")
        ])

        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.classifier = nn.Dense(512, num_classes)

    def _make_layer(self, out_channels, block_count, stride):
        blocks = [
            BasicBlock(self.current_channels, out_channels, stride=stride)
        ]

        self.current_channels = out_channels

        for _ in range(1, block_count):
            blocks.append(
                BasicBlock(self.current_channels, out_channels, stride=1)
            )

        return nn.SequentialCell(blocks)

    def construct(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avg_pool(x)
        x = self.flatten(x)
        return self.classifier(x)


resnet18 = ResNet18(num_classes=2)
resnet_params = sum(int(np.prod(p.shape)) for p in resnet18.trainable_params())

print("ResNet18创建完成")
print("可训练参数量：", resnet_params)
print("可训练参数量（百万）：", f"{resnet_params / 1_000_000:.2f} M")
print("预训练权重：未使用")

## 5. 训练与验证

最终实验配置：

| 模型 | Epoch | 学习率 | Momentum | Weight Decay | 梯度裁剪 |
|---|---:|---:|---:|---:|---:|
| SimpleCNN | 2 | 1e-4 | 0.9 | 1e-4 | 无 |
| ResNet18 | 5 | 1e-3 | 0.9 | 1e-4 | 5.0 |

两种模型均采用随机初始化。

In [7]:
from pathlib import Path
import json
import time
import numpy as np
import mindspore as ms
from mindspore import nn, ops

BATCH_SIZE = globals().get("BATCH_SIZE", 32)

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path("/home/ma-user/work/fake_face_project")

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

if "RESULT_DIR" not in globals():
    RESULT_DIR = PROJECT_DIR / "results"

if "TRAIN_DIR" not in globals():
    SOURCE_ROOT = PROJECT_DIR / "dataset_raw" / "real_vs_fake" / "real-vs-fake"
    TRAIN_DIR = SOURCE_ROOT / "train"
    VAL_DIR = SOURCE_ROOT / "valid"
    TEST_DIR = SOURCE_ROOT / "test"

def evaluate_model(model, dataset_dir, loss_fn):
    if "create_dataset" not in globals():
        raise RuntimeError("请先运行“数据读取与预处理”代码单元。")

    eval_dataset, expected_samples = create_dataset(
        dataset_dir,
        training=False,
        batch_size=BATCH_SIZE
    )

    model.set_train(False)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in eval_dataset.create_dict_iterator(num_epochs=1):
        logits = model(batch["image"])
        loss = loss_fn(logits, batch["label"])

        labels_np = batch["label"].asnumpy()
        predictions = np.argmax(logits.asnumpy(), axis=1)
        n = len(labels_np)

        total_loss += float(loss.asnumpy()) * n
        total_correct += int(np.sum(predictions == labels_np))
        total_samples += n

    return {
        "loss": float(total_loss / total_samples),
        "accuracy": float(total_correct / total_samples),
        "correct": int(total_correct),
        "samples": int(total_samples),
        "expected_samples": int(expected_samples)
    }


def train_network(
    model,
    model_name,
    epochs,
    learning_rate,
    checkpoint_dir,
    result_dir,
    clip_norm=None
):
    if "create_dataset" not in globals():
        raise RuntimeError("请先运行“数据读取与预处理”代码单元。")

    checkpoint_dir = Path(checkpoint_dir)
    result_dir = Path(result_dir)

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    loss_fn = nn.CrossEntropyLoss()

    optimizer = nn.Momentum(
        params=model.trainable_params(),
        learning_rate=learning_rate,
        momentum=0.9,
        weight_decay=1e-4
    )

    weights = model.trainable_params()

    def forward_fn(images, labels):
        logits = model(images)
        loss = loss_fn(logits, labels)
        return loss, logits

    grad_fn = ms.value_and_grad(
        forward_fn,
        grad_position=None,
        weights=weights,
        has_aux=True
    )

    def train_step(images, labels):
        (loss, logits), gradients = grad_fn(images, labels)

        if clip_norm is not None:
            gradients = ops.clip_by_global_norm(
                gradients,
                clip_norm=clip_norm
            )

        optimizer(gradients)
        return loss, logits

    history = {
        "epoch": [],
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "epoch_time_seconds": []
    }

    best_val_accuracy = -1.0
    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        start_time = time.time()

        train_dataset, train_sample_count = create_dataset(
            TRAIN_DIR,
            training=True,
            batch_size=BATCH_SIZE
        )

        steps_per_epoch = train_dataset.get_dataset_size()

        model.set_train(True)

        loss_sum = 0.0
        correct = 0
        samples = 0
        completed_steps = 0

        print(f"\n{model_name} Epoch [{epoch}/{epochs}]")
        print(f"训练样本：{train_sample_count}，批次数：{steps_per_epoch}")

        for step, batch in enumerate(
            train_dataset.create_dict_iterator(num_epochs=1),
            start=1
        ):
            loss, logits = train_step(
                batch["image"],
                batch["label"]
            )

            loss_value = float(loss.asnumpy())

            if not np.isfinite(loss_value):
                emergency_path = (
                    checkpoint_dir
                    / f"{model_name}_abnormal.ckpt"
                )

                ms.save_checkpoint(
                    model,
                    str(emergency_path)
                )

                raise FloatingPointError(
                    f"Epoch {epoch}, Step {step} "
                    f"出现异常Loss：{loss_value}"
                )

            labels_np = batch["label"].asnumpy()
            predictions = np.argmax(
                logits.asnumpy(),
                axis=1
            )

            loss_sum += loss_value
            correct += int(
                np.sum(
                    predictions == labels_np
                )
            )
            samples += len(labels_np)
            completed_steps += 1

            if (
                step == 1
                or step % 100 == 0
                or step == steps_per_epoch
            ):
                print(
                    f"Step [{step:04d}/{steps_per_epoch:04d}] "
                    f"Loss={loss_value:.6f} "
                    f"AvgLoss={loss_sum / completed_steps:.6f} "
                    f"Acc={correct / samples * 100:.2f}%"
                )

        train_loss = loss_sum / completed_steps
        train_accuracy = correct / samples

        val_result = evaluate_model(
            model,
            VAL_DIR,
            loss_fn
        )

        val_loss = val_result["loss"]
        val_accuracy = val_result["accuracy"]
        epoch_time = time.time() - start_time

        print(
            f"Train Loss={train_loss:.6f} | "
            f"Train Acc={train_accuracy * 100:.2f}%"
        )
        print(
            f"Val Loss={val_loss:.6f} | "
            f"Val Acc={val_accuracy * 100:.2f}%"
        )

        ms.save_checkpoint(
            model,
            str(
                checkpoint_dir
                / f"{model_name}_epoch_{epoch:02d}.ckpt"
            )
        )

        ms.save_checkpoint(
            model,
            str(
                checkpoint_dir
                / f"{model_name}_last.ckpt"
            )
        )

        is_better = (
            val_accuracy > best_val_accuracy
            or (
                val_accuracy == best_val_accuracy
                and val_loss < best_val_loss
            )
        )

        if is_better:
            best_val_accuracy = val_accuracy
            best_val_loss = val_loss

            ms.save_checkpoint(
                model,
                str(
                    checkpoint_dir
                    / f"{model_name}_best.ckpt"
                )
            )

        history["epoch"].append(int(epoch))
        history["train_loss"].append(float(train_loss))
        history["train_accuracy"].append(float(train_accuracy))
        history["val_loss"].append(float(val_loss))
        history["val_accuracy"].append(float(val_accuracy))
        history["epoch_time_seconds"].append(float(epoch_time))

        with open(
            result_dir / "training_history.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                history,
                f,
                ensure_ascii=False,
                indent=4
            )

    return history

In [ ]:
# 提交版默认关闭训练。
# 只有真正需要从头训练时，才手动改成 True。
RUN_SIMPLECNN_TRAINING = False
RUN_RESNET18_TRAINING = False

if RUN_SIMPLECNN_TRAINING:
    missing = [
        name for name in [
            "SimpleCNN",
            "train_network",
            "CHECKPOINT_DIR",
            "RESULT_DIR"
        ]
        if name not in globals()
    ]

    if missing:
        print("SimpleCNN训练未启动，缺少前置定义：", missing)
    else:
        model = SimpleCNN(num_classes=2)

        simple_history = train_network(
            model=model,
            model_name="simple_cnn",
            epochs=2,
            learning_rate=1e-4,
            checkpoint_dir=CHECKPOINT_DIR / "simple_cnn",
            result_dir=RESULT_DIR / "simple_cnn",
            clip_norm=None
        )
else:
    print("SimpleCNN训练已关闭。")


if RUN_RESNET18_TRAINING:
    missing = [
        name for name in [
            "ResNet18",
            "train_network",
            "CHECKPOINT_DIR",
            "RESULT_DIR"
        ]
        if name not in globals()
    ]

    if missing:
        print("ResNet18训练未启动，缺少前置定义：", missing)
    else:
        model = ResNet18(num_classes=2)

        resnet_history = train_network(
            model=model,
            model_name="resnet18",
            epochs=5,
            learning_rate=1e-3,
            checkpoint_dir=CHECKPOINT_DIR / "resnet18",
            result_dir=RESULT_DIR / "resnet18",
            clip_norm=5.0
        )
else:
    print("ResNet18训练已关闭。")

## 6. 最佳模型加载与测试

最佳 checkpoint 统一从 AtomGit 仓库 `experiment9` 获取。资源准备单元会下载：

- `resnet18_best.ckpt`
- `simple_cnn_best.ckpt`

因此只验证最终结果时无需重新训练；测试单元优先加载 AtomGit 下载的 `resnet18_best.ckpt`，如果该变量不存在才回退到原项目 `checkpoints/` 目录。

评价指标包括 Accuracy、Precision、Recall、F1-score、Specificity、ROC-AUC 和混淆矩阵，其中 `Fake=1` 为正类。


In [ ]:
from pathlib import Path
import numpy as np
import mindspore as ms
from mindspore import ops

BATCH_SIZE = globals().get("BATCH_SIZE", 32)

if "PROJECT_DIR" not in globals():
    PROJECT_DIR = Path("/home/ma-user/work/fake_face_project")

if "CHECKPOINT_DIR" not in globals():
    CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

if "TEST_DIR" not in globals():
    SOURCE_ROOT = (
        PROJECT_DIR
        / "dataset_raw"
        / "real_vs_fake"
        / "real-vs-fake"
    )
    TEST_DIR = SOURCE_ROOT / "test"

def first_existing(paths):
    for p in paths:
        if p is None:
            continue
        p = Path(p)
        if p.exists() and p.is_file():
            return p
    return None

# 优先使用 AtomGit experiment9 下载的最佳权重；
# 只有未运行资源准备单元时，才回退到原项目 checkpoints/。
RESNET_BEST_CKPT = first_existing([
    globals().get("ATOMGIT_RESNET_CKPT"),
    CHECKPOINT_DIR / "resnet18" / "resnet18_best.ckpt",
    CHECKPOINT_DIR / "resnet18_scratch" / "resnet18_best.ckpt",
    CHECKPOINT_DIR / "resnet18_best.ckpt",
])

SIMPLECNN_BEST_CKPT = first_existing([
    globals().get("ATOMGIT_SIMPLECNN_CKPT"),
    CHECKPOINT_DIR / "simple_cnn" / "simple_cnn_best.ckpt",
    CHECKPOINT_DIR / "simple_cnn_best.ckpt",
])

print("ResNet18最佳权重：", RESNET_BEST_CKPT)
print("SimpleCNN最佳权重：", SIMPLECNN_BEST_CKPT)

def binary_auc_with_ties(labels, scores):
    labels = np.asarray(labels).astype(np.int32)
    scores = np.asarray(scores).astype(np.float64)

    order = np.argsort(scores)
    sorted_scores = scores[order]
    n = len(scores)

    ranks_sorted = np.empty(n, dtype=np.float64)

    i = 0
    while i < n:
        j = i + 1

        while (
            j < n
            and sorted_scores[j] == sorted_scores[i]
        ):
            j += 1

        ranks_sorted[i:j] = (
            ((i + 1) + j) / 2.0
        )

        i = j

    ranks = np.empty(
        n,
        dtype=np.float64
    )

    ranks[order] = ranks_sorted

    positive = labels == 1
    n_pos = int(np.sum(positive))
    n_neg = int(n - n_pos)

    if n_pos == 0 or n_neg == 0:
        return float("nan")

    rank_sum_pos = float(
        np.sum(ranks[positive])
    )

    return float(
        (
            rank_sum_pos
            - n_pos * (n_pos + 1) / 2.0
        )
        / (n_pos * n_neg)
    )

def evaluate_test_set(model, dataset_dir):
    if "create_dataset" not in globals():
        raise RuntimeError(
            "请先运行“数据读取与预处理”代码单元。"
        )

    test_dataset, expected_samples = create_dataset(
        dataset_dir,
        training=False,
        batch_size=BATCH_SIZE
    )

    model.set_train(False)
    softmax = ops.Softmax(axis=1)

    y_true_all = []
    y_pred_all = []
    y_score_all = []

    total_steps = (
        test_dataset.get_dataset_size()
    )

    for step, batch in enumerate(
        test_dataset.create_dict_iterator(
            num_epochs=1
        ),
        start=1
    ):
        logits = model(
            batch["image"]
        )

        probabilities = (
            softmax(logits).asnumpy()
        )

        y_true_all.append(
            batch["label"].asnumpy()
        )

        y_pred_all.append(
            np.argmax(
                logits.asnumpy(),
                axis=1
            )
        )

        y_score_all.append(
            probabilities[:, 1]
        )

        if (
            step == 1
            or step % 100 == 0
            or step == total_steps
        ):
            print(
                f"已处理："
                f"{step:04d}/{total_steps:04d}"
            )

    y_true = np.concatenate(
        y_true_all
    ).astype(np.int32)

    y_pred = np.concatenate(
        y_pred_all
    ).astype(np.int32)

    y_score = np.concatenate(
        y_score_all
    ).astype(np.float64)

    tn = int(np.sum(
        (y_true == 0)
        & (y_pred == 0)
    ))

    fp = int(np.sum(
        (y_true == 0)
        & (y_pred == 1)
    ))

    fn = int(np.sum(
        (y_true == 1)
        & (y_pred == 0)
    ))

    tp = int(np.sum(
        (y_true == 1)
        & (y_pred == 1)
    ))

    accuracy = (
        (tp + tn)
        / len(y_true)
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if tn + fp > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    auc = binary_auc_with_ties(
        y_true,
        y_score
    )

    return {
        "samples": int(len(y_true)),
        "expected_samples": int(expected_samples),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "specificity": float(specificity),
        "roc_auc": float(auc),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

In [ ]:
RUN_CHECKPOINT_TEST = True

required = [
    "ResNet18",
    "evaluate_test_set",
    "TEST_DIR",
    "RESNET_BEST_CKPT"
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    print(
        "checkpoint测试未执行，"
        "请先运行前面的必要单元。缺少：",
        missing
    )

elif not RUN_CHECKPOINT_TEST:
    print("checkpoint测试已关闭。")

elif not TEST_DIR.exists():
    print("checkpoint测试未执行：测试集不存在：", TEST_DIR)

elif RESNET_BEST_CKPT is None:
    print("checkpoint测试未执行：未找到 resnet18_best.ckpt。")

else:
    best_resnet = ResNet18(
        num_classes=2
    )

    params = ms.load_checkpoint(
        str(RESNET_BEST_CKPT)
    )

    not_loaded, not_used = (
        ms.load_param_into_net(
            best_resnet,
            params
        )
    )

    print("未加载参数：", not_loaded)
    print("未使用参数：", not_used)

    if (
        len(not_loaded) == 0
        and len(not_used) == 0
    ):
        metrics = evaluate_test_set(
            best_resnet,
            TEST_DIR
        )

        print("\nResNet18测试集结果")
        print(
            f"Accuracy：    "
            f"{metrics['accuracy'] * 100:.2f}%"
        )
        print(
            f"Precision：   "
            f"{metrics['precision'] * 100:.2f}%"
        )
        print(
            f"Recall：      "
            f"{metrics['recall'] * 100:.2f}%"
        )
        print(
            f"F1-score：    "
            f"{metrics['f1_score'] * 100:.2f}%"
        )
        print(
            f"Specificity： "
            f"{metrics['specificity'] * 100:.2f}%"
        )
        print(
            f"ROC-AUC：     "
            f"{metrics['roc_auc']:.4f}"
        )
        print(
            "TN / FP / FN / TP：",
            metrics["tn"],
            metrics["fp"],
            metrics["fn"],
            metrics["tp"]
        )

## 7. 已完成实验结果与可视化

### 测试集结果

| 模型 | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| SimpleCNN | 62.42% | 65.59% | 52.27% | 58.18% | 0.6730 |
| ResNet18 | **88.61%** | **91.22%** | **85.45%** | **88.24%** | **0.9572** |

ResNet18 相比 SimpleCNN 的测试准确率提升 **26.19 个百分点**。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RESNET_HISTORY_RECORDED = {
    "epoch": [1, 2, 3, 4, 5],
    "train_loss": [0.561854, 0.395312, 0.275586, 0.180198, 0.124136],
    "train_accuracy": [0.7037, 0.8218, 0.8846, 0.9290, 0.9533],
    "val_loss": [0.443624, 0.392930, 0.436982, 0.409971, 0.390299],
    "val_accuracy": [0.7907, 0.8282, 0.8359, 0.8648, 0.8867]
}

SIMPLE_TEST = {
    "accuracy": 62.42,
    "precision": 65.59,
    "recall": 52.27,
    "f1": 58.18,
    "auc": 67.30,
    "cm": [[7258, 2742], [4773, 5227]]
}

RESNET_TEST = {
    "accuracy": 88.61,
    "precision": 91.22,
    "recall": 85.45,
    "f1": 88.24,
    "auc": 95.72,
    "cm": [[9178, 822], [1455, 8545]]
}

epochs = RESNET_HISTORY_RECORDED["epoch"]

plt.figure(figsize=(8, 5))
plt.plot(epochs, RESNET_HISTORY_RECORDED["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(epochs, RESNET_HISTORY_RECORDED["val_accuracy"], marker="s", label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ResNet18 Training and Validation Accuracy")
plt.xticks(epochs)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, RESNET_HISTORY_RECORDED["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs, RESNET_HISTORY_RECORDED["val_loss"], marker="s", label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet18 Training and Validation Loss")
plt.xticks(epochs)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if "SIMPLE_TEST" not in globals():
    SIMPLE_TEST = {
        "accuracy": 62.42,
        "precision": 65.59,
        "recall": 52.27,
        "f1": 58.18,
        "auc": 67.30,
        "cm": [[7258, 2742], [4773, 5227]]
    }

if "RESNET_TEST" not in globals():
    RESNET_TEST = {
        "accuracy": 88.61,
        "precision": 91.22,
        "recall": 85.45,
        "f1": 88.24,
        "auc": 95.72,
        "cm": [[9178, 822], [1455, 8545]]
    }

metric_names = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"
]

simple_values = [
    SIMPLE_TEST["accuracy"],
    SIMPLE_TEST["precision"],
    SIMPLE_TEST["recall"],
    SIMPLE_TEST["f1"],
    SIMPLE_TEST["auc"]
]

resnet_values = [
    RESNET_TEST["accuracy"],
    RESNET_TEST["precision"],
    RESNET_TEST["recall"],
    RESNET_TEST["f1"],
    RESNET_TEST["auc"]
]

x = np.arange(
    len(metric_names)
)

width = 0.36

plt.figure(
    figsize=(10, 5)
)

plt.bar(
    x - width / 2,
    simple_values,
    width,
    label="SimpleCNN"
)

plt.bar(
    x + width / 2,
    resnet_values,
    width,
    label="ResNet18"
)

plt.xticks(
    x,
    metric_names
)

plt.ylabel("Score (%)")
plt.title("SimpleCNN vs ResNet18")
plt.ylim(0, 100)
plt.legend()
plt.grid(
    axis="y",
    alpha=0.25
)
plt.tight_layout()
plt.show()

def show_cm(matrix, title):
    matrix = np.asarray(
        matrix
    )

    plt.figure(
        figsize=(6, 5)
    )

    plt.imshow(matrix)
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")

    plt.xticks(
        [0, 1],
        ["Real", "Fake"]
    )

    plt.yticks(
        [0, 1],
        ["Real", "Fake"]
    )

    for r in range(2):
        for c in range(2):
            plt.text(
                c,
                r,
                str(matrix[r, c]),
                ha="center",
                va="center",
                fontsize=14
            )

    plt.colorbar()
    plt.tight_layout()
    plt.show()

show_cm(
    SIMPLE_TEST["cm"],
    "SimpleCNN Confusion Matrix"
)

show_cm(
    RESNET_TEST["cm"],
    "ResNet18 Confusion Matrix"
)

## 8. 实验结论

- SimpleCNN 测试准确率为 **62.42%**，Fake 类召回率为 **52.27%**，适合作为基础基线。
- ResNet18 在**不使用预训练权重**的情况下，从头训练 5 个 Epoch 后，测试准确率达到 **88.61%**。
- ResNet18 的 F1-score 达到 **88.24%**，ROC-AUC 达到 **0.9572**，整体区分能力显著强于 SimpleCNN。
- 最佳验证准确率为 **88.67%**，测试准确率为 **88.61%**，二者接近。
- ResNet18 仍有 **1,455** 张 Fake 被误判为 Real，后续改进重点是降低伪造人脸漏检。

### 后续方向

1. 使用 ImageNet 预训练 ResNet18 进行迁移学习；
2. 增加高频残差 / 频域特征分支；
3. 使用学习率衰减并增加训练轮数；
4. 在其他 Deepfake 数据集上开展跨数据集验证。

> 本实验结果代表当前数据集上的测试表现，不直接等同于所有未知伪造人脸场景下的性能。

## 9. 提交目录

```text
fake_face_detection_submission/
├── code/
│   └── fake_face_detection_final.ipynb
├── dataset/
│   └── real_vs_fake/
│       └── real-vs-fake/
│           ├── train/
│           │   ├── real/
│           │   └── fake/
│           ├── valid/
│           │   ├── real/
│           │   └── fake/
│           └── test/
│               ├── real/
│               └── fake/
├── checkpoints/
│   ├── simple_cnn/
│   │   └── simple_cnn_best.ckpt
│   └── resnet18/
│       └── resnet18_best.ckpt
└── results/
    ├── simple_cnn/
    └── resnet18/
```

**复现建议：**

- 只验证最终结果：保持训练开关为 `False`，直接加载最佳 checkpoint。
- 完整复现训练：在 Ascend 环境下将对应 `RUN_*_TRAINING` 设置为 `True`。
**AtomGit 资源说明：**

- 数据集：`experiment9/archive.zip`，首次运行自动下载并解压；
- ResNet18 最佳模型：`experiment9/resnet18_best.ckpt`；
- SimpleCNN 最佳模型：`experiment9/simple_cnn_best.ckpt`；
- 仓库页面：`https://ai.gitcode.com/qq_627548571/face/tree/main/experiment9`。

本地 `dataset/` 与 `checkpoints/` 结构仍保留兼容性，但在运行本 Notebook 时优先使用 AtomGit 下载的资源。

